# Reproducing Knierim, McNaughton & Poe (2000) from dandiset 001754

Paper. Knierim JJ, McNaughton BL, Poe GR (2000). Three-dimensional spatial selectivity of
hippocampal neurons during space flight. *Nature Neuroscience* 3(3):211-212.
[doi:10.1038/72910](https://doi.org/10.1038/72910). The claims file,
`claims/001754/10.1038-72910.yaml`, was written from the full text of the version of record, a
two-page brief communication with no separate Methods section and no supplement; its methods are
in the main text and the figure legends. The paper links no code.

Dandiset. [DANDI:001754](https://dandiarchive.org/dandiset/001754/0.260728.1352), version
0.260728.1352 (doi:10.48324/dandi.001754/0.260728.1352). Why this version: the paper cites no
version of the dandiset, so the most recent published version is pinned, and 0.260728.1352 is it.
The two earlier published versions (0.260227.0029 and 0.260227.0031) hold the same files with the
same blobs (data check), so none of them matches the claims file better. The draft holds a new
conversion that addresses the findings of the first attempt, but it is not published, and it has
changed since `001754-knierim-2000-02` read it (the data check prints when it was last
modified), so it is not used. This is the version that `001754-knierim-2000-01` read;
`001754-knierim-2000-02` read the draft.

This attempt, `001754-knierim-2000-03`, replaces `001754-knierim-2000-02`, whose claims were
written from the abstract and the figure titles. The claims file has been written again from the
full text. Three claims keep their ids with criteria rewritten from the paper
(`flight-spatial-code`, now judged by comparing spatial information with preflight,
`fig3-stability-across-sessions` and `three-orthogonal-surfaces`), four return
(`si-comparable-to-preflight`, `fd4-abnormal-selectivity`, `theta-normal`, `ripples-normal`), and
three are new (`si-rat1-fd9-vs-preflight`, `si-rat3-fd4-vs-preflight`, `direction-selectivity`).
The claims file names readings of the map (M-a, the deposited maps; M-b, maps recomputed from
spikes and position) and of the session (S-a, each candidate; S-b, the one closest before launch).
How each applies to these files is said under the claims, and what the claims file does not settle
is asked in `questions/001754.yaml`.

What the dandiset holds. Three rats on the Neurolab mission (STS-90, April 1998): one preflight
recording of each, flight day 4 (FD4) of each, and flight day 9 (FD9) of Rats 1 and 2. Each file
has sorted units, task epochs (BL baseline, ES Escher staircase, MC Magic Carpet) and the 64 x 64
rate and occupancy maps of the original analysis for each ES and MC epoch. The flight files also
hold the video position, at spike times only. There is no EEG.

| Claim | Source in the paper | Tested |
|---|---|---|
| `flight-spatial-code`: normal spatially specific firing in flight, spatial information as preflight | Main text and Figs. 2a, 2c, 3a | Yes, from the three comparisons below |
| `si-comparable-to-preflight`: rat 2, FD9 against preflight | Main text, Fig. 2a,b | Yes |
| `si-rat1-fd9-vs-preflight`: rat 1, FD9 against preflight | Main text, Fig. 2c | Yes |
| `si-rat3-fd4-vs-preflight`: rat 3, FD4 session 2 against preflight | Main text, Fig. 3a | Yes |
| `fd4-abnormal-selectivity`: abnormal selectivity of rats 1 and 2 on FD4 | Main text | Yes |
| `fig3-stability-across-sessions`: fields kept from session 2 to session 3 | Main text, Fig. 3 | Yes |
| `direction-selectivity`: counterclockwise-only place fields | Main text | Blocked: data missing |
| `theta-normal`: normal theta in flight | Main text | Blocked: data missing |
| `ripples-normal`: normal sharp waves and ripples in flight | Main text | Blocked: data missing |
| `three-orthogonal-surfaces`: representations on the three surfaces | Main text, standfirst | Blocked |

In [ ]:
import json
import re
import time
import warnings
import zipfile
from collections import Counter

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
import requests
from scipy import stats

import neurolab as nl

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

EARLIER = "001754-knierim-2000-02"
EARLIER_DRAFT_MODIFIED = "2026-10-14T21:55:09Z"  # the draft as 001754-knierim-2000-02 read it (its record)
started = time.time()
sessions = nl.load_all()
by_label = {nl.label(s): s for s in sessions}
print(f"loaded {len(sessions)} files from dandiset {nl.DANDISET_ID} version {nl.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output. The status of each
finding of `001754-knierim-2000-02` on this version is given at the end of the data check.

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/001754/versions"
listing = requests.get(f"{api}/", timeout=60).json()["results"]
print("Versions and when each was last modified:")
for v in listing:
    print(f"  {v['version']:15s} {v['modified']}")
draft_modified = next(v["modified"] for v in listing if v["version"] == "draft")
print(f"The draft was last modified {draft_modified}; {EARLIER} read it as last modified {EARLIER_DRAFT_MODIFIED}.")
blobs = {}
for v in [x["version"] for x in listing]:
    assets = requests.get(f"{api}/{v}/assets/", params={"page_size": 100}, timeout=60).json()["results"]
    blobs[v] = sorted((a["path"], a["size"], a.get("blob")) for a in assets)
published = [v for v in blobs if v != "draft"]
print("\nThe published versions hold the same files with the same blobs:", all(blobs[v] == blobs[published[0]] for v in published))
pinned = {p: b for p, _, b in blobs[nl.DANDISET_VERSION]}
print(f"Files of the draft that are a different blob from {nl.DANDISET_VERSION}:",
      sum(b != pinned.get(p) for p, _, b in blobs["draft"]), "of", len(blobs["draft"]))
meta = {v: requests.get(f"{api}/{v}/", timeout=60).json() for v in ["draft", nl.DANDISET_VERSION]}
print(f"DOI of {nl.DANDISET_VERSION}:", meta[nl.DANDISET_VERSION].get("doi"))
print(f"The dandiset description is the same in the draft and {nl.DANDISET_VERSION}:",
      meta["draft"]["description"] == meta[nl.DANDISET_VERSION]["description"])
print("\nDandiset description:\n", meta[nl.DANDISET_VERSION]["description"])
print("\nSentences of the draft's description that are not in the pinned version's:")
for sentence in meta["draft"]["description"].split(". "):
    if sentence not in meta[nl.DANDISET_VERSION]["description"]:
        print("  ", sentence.strip())

print("\nEvery group (with its neurodata type) and dataset (shape, type, description) of every file, unfiltered:")
types = {}
for path_url in nl.asset_urls():
    tree = nl.h5_tree(path_url)
    print(f"\n{path_url[0]}: {len(tree)} groups and datasets")
    for line in tree:
        print(" ", line)
        if "  group  " in line and line.split("  group  ")[1]:
            types.setdefault(line.split("  group  ")[1], set()).add(path_url[0])
print("\nNeurodata types of groups, and the number of files holding each:", {t: len(v) for t, v in sorted(types.items())})

The three published versions hold the same eight files with the same blobs; every file of the draft
is a different blob, and the draft's description has been rewritten (the sentences printed above:
it no longer places the preflight recordings on a flat rectangular track, and it says the EEG is
not included). Those changes are in the draft only; the pinned version keeps the description
printed above. The listing shows every group and dataset of every file. The neurodata types are
only those printed under it: there is no ElectricalSeries or LFP, so no file holds EEG or any
continuous voltage. The units table has `tetrode`, `cluster_id`, `spike_times` and `electrodes`;
the epochs table `session_type` and its description; the rate-map table `tetrode`, `source_file`,
`session_type`, `cell_number`, `rate_map` and `occupancy_map`; the position one series of x and y.

In [ ]:
# The descriptions of the tables, columns and series the analysis uses, from one flight file and
# one preflight file (the others carry the same text), and every subject's age and description.
for path_url in [p for p in nl.asset_urls() if "ses-19980425T" in p[0] and "Rat1" in p[0]] + [p for p in nl.asset_urls() if "ses-19980413T" in p[0]]:
    print(f"== {path_url[0]}")
    with h5py.File(remfile.File(path_url[1]), "r") as f:
        for key in ["general/experiment_description", "general/session_id", "general/keywords"]:
            if key in f:
                print(f"{key}: {f[key][()]!r}")
        for name in ["units", "intervals/epochs", "processing/ecephys/rate_maps", "processing/behavior/position/spatial_series"]:
            if name not in f:
                print(f"{name}: not in this file")
                continue
            g = f[name]
            print(f"{name}: {nl._text(g.attrs.get('description', ''))!r}")
            for col in g.keys():
                if isinstance(g[col], h5py.Dataset):
                    note = {k: nl._text(v) for k, v in g[col].attrs.items() if k in ("description", "unit", "resolution", "conversion")}
                    print(f"    {col}: {note}")
    print()
for p, url in nl.asset_urls():
    with h5py.File(remfile.File(url), "r") as f:
        print(f"{p.split('/')[1][:31]}: age {nl._text(f['general/subject/age'][()])!r}; {nl._text(f['general/subject/description'][()])!r}")

The rate-map table says a map with `cell_number` other than -1 can be matched to the unit with
that cluster id on the same tetrode, and that the rate is in Hz; it does not say the maps are
smoothed or how unvisited bins are coded. The position is described as sampled at spike times, in
pixels, with no camera view or pixel scale. Every subject's age is "/". The experiment description
says baseline sessions were on a rectangular track, interleaved with task sessions; the paper
says baseline sessions were in the sleeping pouch (EEG paragraph) and that rat 2 ran a flat
rectangular track preflight (Fig. 2b). No file of the pinned version describes the preflight
apparatus.

In [ ]:
rows = []
for s in sessions:
    ep = s["epochs"]
    iv = sorted(zip(ep.start, ep.stop))
    rows.append({"file": s["path"].split("/")[1][:31], "rat": s["subject"], "day": s["day"], "start": s["start"][:16],
                 "units": len(s["units"]), "spikes": int(sum(len(x) for x in s["units"]["spikes"])),
                 "tetrodes": s["units"]["tetrode"].nunique(),
                 "epochs": " ".join(f"{n}[{a:.0f}-{b:.0f}]" for a, b, n in ep[["start", "stop", "name"]].itertuples(index=False)),
                 "overlapping_epochs": sum(1 for (a1, b1), (a2, b2) in zip(iv, iv[1:]) if a2 < b1),
                 "rate_maps": len(s["maps"]), "position_samples": len(s["t"]) if "t" in s else 0})
files = pd.DataFrame(rows)
print(files.to_string(index=False))
print("\nFiles per rat and day:"); print(pd.crosstab(files.rat, files.day).to_string())
for s in sessions:
    print(f"{nl.label(s):15s} {s['description']}")
print("\nTetrodes with units, per file:")
for s in sessions:
    print(f"  {nl.label(s):15s}", sorted(s["units"].tetrode.unique(), key=lambda t: int(t[2:])))
for rat in ["Rat1", "Rat2", "Rat3"]:
    pre = set(by_label[f"{rat} preflight"]["units"].tetrode)
    for day in ["FD4", "FD9"]:
        if f"{rat} {day}" in by_label:
            print(f"{rat}: tetrodes in both preflight and {day}: {len(pre & set(by_label[f'{rat} {day}']['units'].tetrode))}")

Three rats, eight files: preflight and FD4 for each rat, FD9 for Rats 1 and 2. The preflight files
were recorded 3 days (Rats 1 and 3) and 4 days (Rat 2) before launch, as their descriptions say;
Rat 2's matches the paper's "4 days before launch" (Fig. 2b). Each preflight file holds one ES and
one MC epoch and no baseline. Rat3 FD4 has two ES epochs, ES1 (32 to 1345 s) and ES2 (2397 to 3729
s); the paper says rat 3's first FD4 session was lost and its sessions 2 and 3 followed one another,
so ES1 and ES2 are read as sessions 2 and 3. Rat1 FD4 has two baseline epochs that overlap (2525 to
3390 s and 2720 to 3390 s). Rat 2's preflight units are on tetrodes TT0 to TT10 and its flight
units on TT12 to TT22, with no tetrode in common, while Rats 1 and 3 share 4 to 6 tetrodes between
preflight and flight; nothing in the files says why, so the per-rat comparisons with preflight
below assume the preflight files are assigned to the right rats.

In [ ]:
print("Position, flight files. Stated unit:", {s['xy_unit'] for s in sessions if 'xy' in s}, "; description:", {s['xy_description'] for s in sessions if 'xy' in s})
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    xy, v, t = s["xy"], s["valid"], s["t"]
    own = np.unique(np.concatenate(list(s["units"]["spikes"])))
    row = {"session": nl.label(s), "samples": len(t), "lost_(0,255)": round(1 - v.mean(), 3),
           "x_range": f"{xy[v, 0].min():.0f}-{xy[v, 0].max():.0f}", "y_range": f"{xy[v, 1].min():.0f}-{xy[v, 1].max():.0f}",
           "t_range_s": f"{t[0]:.0f}-{t[-1]:.0f}", "at_own_spike_times": round(float(np.isin(t, own).mean()), 4)}
    for typ in ["ES", "MC"]:
        for a, b, n in s["epochs"][s["epochs"].type == typ][["start", "stop", "name"]].itertuples(index=False):
            sp = own[(own >= a) & (own <= b)]
            i = np.clip(np.searchsorted(t, sp), 1, len(t) - 1)
            has = np.minimum(np.abs(t[i - 1] - sp), np.abs(t[i] - sp)) <= nl.LOOKUP_S
            row[f"{n}_spike_times_with_sample"] = round(float(has.mean()), 4)
            if not has.all():
                row[f"{n}_without_sample_s"] = f"{sp[~has].min():.0f}-{sp[~has].max():.0f}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print("\nDo the Rat1 and Rat2 files of a flight day hold the same trace? Timestamps in both files, and of those, the share within 2 px (both valid):")
for day in ["FD4", "FD9"]:
    a, b = by_label[f"Rat1 {day}"], by_label[f"Rat2 {day}"]
    common, ia, ib = np.intersect1d(a["t"], b["t"], return_indices=True)
    both = a["valid"][ia] & b["valid"][ib]
    d = np.hypot(*(a["xy"][ia][both] - b["xy"][ib][both]).T)
    print(f"  {day}: {len(common)} common timestamps ({len(common) / len(a['t']):.3f} of Rat1's, {len(common) / len(b['t']):.3f} of Rat2's); "
          f"within 2 px: {(d < 2).mean():.3f}; Rat1 samples span {a['t'][0]:.0f}-{a['t'][-1]:.0f} s, Rat2 {b['t'][0]:.0f}-{b['t'][-1]:.0f} s")

Position is in video pixels, 0 to 255 in x and 54 to 255 in y. The lost position (0, 255) is 5 to
12% of samples; it is not a position and is dropped. Every sample is at a spike time of the file's
own units, and every spike of an ES epoch has a sample at its own time, so reading M-b can place
every ES spike that is not at the lost position. In Rat3 FD4 the position ends at 6308 s, 41 s
before the end of its MC epoch (5.3% of MC spikes have no sample), and in Rat2 FD9 at 10144 s, 7 s
before the end of its MC epoch; no claim uses MC position. The Rat1 and Rat2 files of a flight day
hold one trace: where they share a timestamp, 98.6 to 98.8% of positions agree within 2 pixels, and
both span the whole session. Each file presents it as its own rat's position, though one tracker
served both rats (the files say the rats were recorded on a shared acquisition system).

In [ ]:
print("Distinct occupancy maps per task type, per file:", {nl.label(s): dict(Counter(t for t, _, _ in nl.occupancy_groups(s))) for s in sessions})
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for typ, key, idx in nl.occupancy_groups(s):
        occ = s["occupancy_maps"][idx[0]]
        assigned = s["maps"].epoch[idx[0]]
        scores = {n: round(nl.occupancy_match(s, occ, a, b), 2) for a, b, n in s["epochs"][["start", "stop", "name"]].itertuples(index=False)}
        a, b = s["epochs"].set_index("name").loc[assigned, ["start", "stop"]]
        m = (s["t"] >= a) & (s["t"] <= b) & s["valid"]
        swapped = nl.count_map(*nl.bin_xy(s["xy"][m][:, ::-1])) > 0
        rows.append({"session": nl.label(s), "maps": len(idx), "assigned_epoch": assigned, "match_assigned": scores[assigned],
                     "match_other_epochs": {k: v for k, v in scores.items() if k != assigned},
                     "match_x_y_swapped": round(float(np.corrcoef((nl.clean_occupancy(occ) > 0).ravel(), swapped.ravel())[0, 1]), 2),
                     "occupancy_counts": int(occ.sum()), "counts_per_s": round(occ.sum() / (b - a), 1)})
print("Correlation of each binary occupancy map with the binary histogram of an epoch's valid position samples:")
print(pd.DataFrame(rows).to_string(index=False))
print("\nEpoch of each map, per file and type:", {nl.label(s): s["maps"].groupby(["type", "epoch"]).size().to_dict() for s in sessions})

Each task type has one occupancy map per file, except Rat3 FD4, which has two ES maps. With
position binned at 4 pixels per bin, x to columns and y to rows, each ES occupancy map matches the
position samples of its own epoch best (0.68 to 0.88), better than the other epochs and better than
with x and y swapped (0.41 to 0.62). The MC maps match their epochs at 0.46 to 0.88, and best in
every file except Rat2 FD9, whose MC map matches the window in which Rat1 ran the staircase (BL1 in
Rat2's file, 0.51) better than its own MC epoch (0.46). Rat3's two ES occupancy maps match ES1
(0.81) and ES2 (0.83), and the maps are assigned to those epochs. Rat2's FD9 ES map matches Rat2's
ES epoch (0.68) nearly as well as Rat1's ES window (0.64), as a shared trace would. The occupancy
counts are 38 to 57 per second of an epoch, the order of a video frame rate; the description calls
them visit counts. The ES epoch names, times and position are therefore consistent with the
original analysis.

In [ ]:
r_all = np.concatenate([s["rate_maps"].ravel() for s in sessions])
o_all = np.concatenate([s["occupancy_maps"].ravel() for s in sessions])
print("Rate-map values: -999 bins", round(float((r_all == nl.UNVISITED).mean()), 3), "; -999 exactly where occupancy is 0:",
      bool(((r_all == nl.UNVISITED) == (o_all == 0)).all()), "; other negative values:", int(((r_all < 0) & (r_all != nl.UNVISITED)).sum()),
      "; NaN:", int(np.isnan(r_all).sum()))
print("Max rate per file (Hz):", {nl.label(s): round(float(s["rate_maps"].max()), 1) for s in sessions})
print("\nOccupancy in bin [63, 0], where the lost position (0, 255) falls, as a fraction of each occupancy map:")
for s in sessions:
    print("  ", nl.label(s), [f"{s['maps'].epoch[idx[0]]} {s['occupancy_maps'][idx[0]][nl.SENTINEL_BIN] / s['occupancy_maps'][idx[0]].sum():.3f}" for _, _, idx in nl.occupancy_groups(s)])

print("\nIdentical copies of a rate map within a file (each group: type, tetrode/short name of every copy):")
for s in sessions:
    groups = {}
    for i, r in enumerate(s["rate_maps"]):
        groups.setdefault(r.tobytes(), []).append(i)
    for rows_ in groups.values():
        if len(rows_) > 1:
            print(f"  {nl.label(s)}: {s['maps'].type[rows_[0]]} " + ", ".join(f"{s['maps'].tetrode[i]}/{s['maps'].source_file[i]}" for i in rows_))
print("Maps counted once per file:", {nl.label(s): int((~s["maps"].duplicate).sum()) for s in sessions})
print("\ncell_number -1 (\"not a per-cell map\"), per file:", {nl.label(s): f"{int((s['maps'].cell_number == -1).sum())} of {len(s['maps'])}" for s in sessions})
print("Rat3 FD4 ES maps with cell_number -1:", int(((by_label['Rat3 FD4']['maps'].type == 'ES') & (by_label['Rat3 FD4']['maps'].cell_number == -1)).sum()),
      "of", int((by_label['Rat3 FD4']['maps'].type == 'ES').sum()), "; their names:", sorted(set(by_label['Rat3 FD4']['maps'].source_file[by_label['Rat3 FD4']['maps'].type == 'ES'])))

In [ ]:
# Can a rate map be linked to its unit? For each flight unit and task epoch with 50 or more spikes at
# a position, the correlation of the unit's spike-count map with rate x occupancy of every distinct map
# of its tetrode and epoch; the best match, and whether the table's stated link (same tetrode and type,
# cell_number = cluster_id) names it.
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ == "BL":
            continue
        cand = s["maps"][(s["maps"].epoch == name) & ~s["maps"].duplicate]
        for ui, u in s["units"].iterrows():
            ok, r, c = nl.spike_positions(s, u.spikes[(u.spikes >= a) & (u.spikes <= b)])
            if ok.sum() < 50:
                continue
            counts = nl.count_map(r, c)
            cc = cand[cand.tetrode == u.tetrode]
            rs = {i: np.corrcoef((np.clip(s["rate_maps"][i], 0, None) * nl.clean_occupancy(s["occupancy_maps"][i])).ravel(), counts.ravel())[0, 1] for i in cc.index}
            rs = {i: v for i, v in rs.items() if np.isfinite(v)}
            stated = list(cc.index[cc.cell_number == u.cluster])
            best = max(rs, key=rs.get) if rs else None
            row = {"session": nl.label(s), "epoch": name, "tetrode": u.tetrode, "cluster": int(u.cluster), "spikes": int(ok.sum()),
                   "maps_on_tetrode": len(cc), "best_r": rs[best] if best is not None else np.nan,
                   "stated_link_exists": bool(stated), "stated_link_is_best": best in stated}
            if best is not None:
                rate_i, occ_i = s["rate_maps"][best], nl.clean_occupancy(s["occupancy_maps"][best])
                positive = (rate_i > 0) & (occ_i > 0)
                row["positive_bins_without_spikes"] = float((positive & (counts == 0)).sum() / max(positive.sum(), 1))
            rows.append(row)
link = pd.DataFrame(rows)
print(f"Unit-epochs with >= 50 spikes at a position: {len(link)}; with a map on their tetrode: {int(link.best_r.notna().sum())}; "
      f"best match r > 0.5: {int((link.best_r > 0.5).sum())}")
print(f"The table's stated link exists for {int(link.stated_link_exists.sum())} and names the best-matching map for {int(link.stated_link_is_best.sum())}")
print(link.groupby(["session", "epoch"]).agg(n=("best_r", "size"), median_best_r=("best_r", "median"), stated_exists=("stated_link_exists", "sum"),
      stated_is_best=("stated_link_is_best", "sum")).round(2).to_string())
good = link[link.best_r > 0.5]
print(f"\nBest-matching maps with r > 0.5 ({len(good)}): fraction of bins with a positive rate where the unit has no spike, median {good.positive_bins_without_spikes.median():.2f}")

In [ ]:
# The helper's measures on synthetic maps: a uniform rate carries no information; a rate confined to
# one of four equally occupied bins carries log2(4) = 2 bits per spike; the peak-in-field test.
occ = np.zeros((64, 64)); occ[:2, :2] = 10
print("uniform:", nl.skaggs_info(np.where(occ > 0, 3.0, 0.0), occ), "; one of four bins:", nl.skaggs_info(np.where(occ > 0, 0.0, 0.0) + (np.arange(64 * 64).reshape(64, 64) == 0) * 5.0, occ))
m1 = np.zeros((64, 64)); m1[0, 0], m1[0, 1] = 5.0, 2.0
m2 = np.zeros((64, 64)); m2[0, 1] = 4.0
m3 = np.zeros((64, 64)); m3[1, 1] = 4.0
print("peak in field (session 3 peak where session 2 has 40% of its peak):", nl.peak_in_field(m1, m2, occ > 0), "; (where it has 0%):", nl.peak_in_field(m1, m3, occ > 0))
assert abs(nl.skaggs_info(np.where(occ > 0, 3.0, 0.0), occ)) < 1e-12 and nl.peak_in_field(m1, m2, occ > 0) and not nl.peak_in_field(m1, m3, occ > 0)

Unvisited bins are -999, exactly where the occupancy is zero; the column description gives the rate
in Hz and does not mention -999. No other value is negative. The lost position is counted in the
original occupancy, in bin [63, 0] (up to 19% of a flight occupancy map, under 2% preflight); that
bin is left out of every map here. Six groups of identical maps are stored in three files, several
under different tetrodes (Rat2 FD4 TT13, TT19 and TT22); each map is counted once.

`cell_number` does not link maps to units as the table says. Of 211 flight unit-epochs with 50 or
more spikes at a position, the stated link exists for 98 and names the map that best matches the
unit's spikes for 84. Every one of Rat3 FD4's 42 ES maps has `cell_number` -1, "not a per-cell
map", though their names (ES1CEL~1 to ES2CEL~4 and four hashed names) are 8.3 short forms of
per-cell file names; so the dandiset gives no way to pair Rat 3's session 2 and session 3 maps by
cell. In the best-matching maps that correlate above 0.5 with their unit's spikes (135), the median
map has 91% of its positive-rate bins without a spike of the unit: the maps are smoothed, with a
kernel not stated. The synthetic tests show the helper's information measure and peak-in-field
test behave as intended.

In [ ]:
ranges = [(0, 1000), (1000, 10000), (10000, np.inf)]
rows = []
for s in sessions:
    allsp = np.concatenate(list(s["units"].spikes))
    for lo, hi in ranges:
        z = n = 0
        for sp in s["units"].spikes:
            w = sp[(sp >= lo) & (sp < hi)]
            n += max(len(w) - 1, 0)
            z += int((np.diff(w) == 0).sum())
        distinct = np.unique(allsp[(allsp >= lo) & (allsp < hi)])
        if len(distinct) > 1:
            rows.append({"session": nl.label(s), "time_range_s": f"{lo}-{hi}", "spikes": int(((allsp >= lo) & (allsp < hi)).sum()),
                         "smallest_step_s": round(float(np.diff(distinct).min()), 4), "zero_intervals": z, "zero_fraction": round(z / max(n, 1), 3)})
print("Spike times are sorted in every unit:", all((np.diff(sp) >= 0).all() for s in sessions for sp in s["units"].spikes))
print(pd.DataFrame(rows).to_string(index=False))
outside = {nl.label(s): [int((~np.any([(sp >= a) & (sp <= b) for a, b in zip(s["epochs"].start, s["epochs"].stop)], axis=0)).sum()) for sp in s["units"].spikes] for s in sessions}
print("\nSpikes outside every epoch of their file (units with any, spikes):", {k: (sum(x > 0 for x in v), sum(v)) for k, v in outside.items()})

Spike times are sorted. They are resolved to 1 ms or finer below 1000 s, to 10 ms from 1000 s and to
0.1 s from 10000 s, and zero intervals within a unit follow that resolution (none or nearly none
below 1000 s, 5 to 15% at 10 ms, 35 to 53% at 0.1 s); nothing in the files states the resolution.
1 to 14 spikes per file lie outside every epoch. The analysis uses spikes within an epoch, so
neither changes a result here except through the 10 ms binning of time, which is far finer than
the 4-pixel bins and the epochs.

In [ ]:
# Shared clock of spikes and position, and whose position the trace is. Each unit with >= 50 spikes at a
# position in a task window: spatial information of its spike-count map over the window's deposited
# occupancy (as in reading M-b) with its real spike times, against the same with its spikes shifted by
# +-30, 60, 90, 120 and 150 s (circularly within the window; each shifted spike takes the nearest sample
# within 0.25 s, else it is dropped). Shifted spikes are dropped more often than real ones, so for each
# shift the real spikes are subsampled to the same count (fixed seed). These are standard choices for a
# check; they decide no claim.
# Every unit is run with its own file's trace in every ES and MC window of both rats of a flight day.
SHIFTS = [s_ * k for k in (30, 60, 90, 120, 150) for s_ in (1, -1)]
rng_clock = np.random.default_rng(1754)

def shift_contrast(s, a, b, occ):
    occ = nl.clean_occupancy(occ)
    t, valid = s["t"], s["valid"]
    out = []
    for _, u in s["units"].iterrows():
        sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
        ok, r, c = nl.spike_positions(s, sp)
        if ok.sum() < 50:
            continue
        def info(rows, cols):
            counts = nl.count_map(rows, cols)
            counts[occ == 0] = 0
            return nl.skaggs_info(np.where(occ > 0, counts / np.where(occ > 0, occ, 1), 0.0), occ) if counts.sum() else np.nan
        real = info(r, c)
        shifted, matched = [], []
        for sh in SHIFTS:
            x = a + (sp - a + sh) % (b - a)
            i = np.clip(np.searchsorted(t, x), 1, len(t) - 1)
            j = np.where(np.abs(t[i - 1] - x) <= np.abs(t[i] - x), i - 1, i)
            keep = (np.abs(t[j] - x) <= 0.25) & valid[j]
            shifted.append(info(*nl.bin_xy(s["xy"][j[keep]])) if keep.sum() else np.nan)
            # the real spikes subsampled to the number of shifted spikes kept, as information rises when spikes are fewer
            pick = rng_clock.choice(len(r), min(int(keep.sum()), len(r)), replace=False)
            matched.append(info(r[pick], c[pick]) if len(pick) else np.nan)
        diff = np.array(matched) - np.array(shifted)
        out.append({"real": real, "shift_median": np.nanmedian(shifted), "above_all": bool(np.all(diff[np.isfinite(diff)] > 0)), "matched_median": np.nanmedian(matched)})
    return pd.DataFrame(out)

rows = []
for day in ["FD4", "FD9"]:
    pair = {r: by_label[f"{r} {day}"] for r in ["Rat1", "Rat2"]}
    for units_of in ["Rat1", "Rat2"]:
        for window_of in ["Rat1", "Rat2"]:
            w = pair[window_of]
            for a, b, typ, name in w["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
                if typ == "BL":
                    continue
                d = shift_contrast(pair[units_of], a, b, nl.epoch_occupancy(w, name))
                rows.append({"day": day, "units_of": units_of, "window": f"{window_of} {name} {a:.0f}-{b:.0f}", "units": len(d),
                             "median_real": round(float(d.real.median()), 3) if len(d) else np.nan,
                             "median_real_count_matched": round(float(d.matched_median.median()), 3) if len(d) else np.nan,
                             "median_shifted": round(float(d.shift_median.median()), 3) if len(d) else np.nan,
                             "above_every_shift": int(d.above_all.sum()) if len(d) else 0,
                             "binom_p_vs_1/11": stats.binomtest(int(d.above_all.sum()), len(d), 1 / 11, alternative="greater").pvalue if len(d) else np.nan})
s3 = by_label["Rat3 FD4"]
for a, b, typ, name in s3["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
    d = shift_contrast(s3, a, b, nl.epoch_occupancy(s3, name))
    rows.append({"day": "FD4", "units_of": "Rat3", "window": f"Rat3 {name} {a:.0f}-{b:.0f}", "units": len(d),
                 "median_real": round(float(d.real.median()), 3),
                 "median_real_count_matched": round(float(d.matched_median.median()), 3), "median_shifted": round(float(d.shift_median.median()), 3),
                 "above_every_shift": int(d.above_all.sum()), "binom_p_vs_1/11": stats.binomtest(int(d.above_all.sum()), len(d), 1 / 11, alternative="greater").pvalue})
contrast = pd.DataFrame(rows)
print("Real against time-shifted spikes: units whose count-matched real information exceeds that of all 10 shifts (1/11 expected by chance):")
print(contrast.to_string(index=False))

This contrast tests whether spikes and position share a clock, and whose position the trace is. In
their own ES windows, units of Rat1 (5 of 20 on FD4, 9 of 25 on FD9), Rat3 (6 of 16 and 5 of 19)
and Rat2 on FD9 (12 of 28) carry more information with their real spike times than with all ten
time shifts, against 1/11 expected by chance (binomial p 0.030, 0.00022, 0.0020, 0.024 and 2e-6).
So for those windows spikes and position share a clock, and in Rat2's FD9 ES window the trace
behaves as Rat2's position; Rat1's units are not selective there (0 of 36). In Rat2's FD4 ES window
neither rat's units are (0 of 17 for Rat2, 1 of 17 for Rat1), so whose position the trace is in that
window, and whether Rat2's FD4 spikes share its clock, could not be established. In MC windows the
contrast is weak for every rat (0 to 3 units). The FD4 results for Rat2 below carry this caveat.
This check counts real spikes subsampled to the shifted count and uses 10 fixed shifts; it differs
from the shuffle test of the earlier attempts, which found Rat2's FD9 units not selective.

### Findings of `001754-knierim-2000-02` on this version

`001754-knierim-2000-02` read the draft. The pinned version holds the files that
`001754-knierim-2000-01` read, so each finding is in the state the first attempt found it, except
where the outputs above show more:

- The Rat1 and Rat2 files hold one shared trace presented as each rat's position (partly fixed in
  the draft). Still present in this version: the files share timestamps whose positions agree within
  2 pixels for 98.6 to 98.8%, both span the whole session, and nothing names the tracked animal. The
  time-shift contrast shows the trace behaves as Rat2's position in Rat2's FD9 ES window (12 of 28
  units), and not in Rat2's FD4 ES window.
- Position only at spike times, none preflight. Still present.
- The dandiset description places the preflight recordings on a flat rectangular track while the
  preflight files label their epochs ES and MC. Still present in this version; the draft's
  description has dropped the sentence. The paper states the rectangular track for rat 2 only, and
  the files' experiment description puts the rectangular track in the baseline sessions, which the
  paper places in the sleeping pouch.
- The rate maps are smoothed with an unstated kernel (partly fixed in the draft). Still present:
  the pinned descriptions do not say the maps are smoothed; 91% of positive-rate bins of a
  well-matched map hold no spike.
- No EEG though the description reports it (partly fixed in the draft). Still present: no file
  holds EEG, and the pinned description reports theta and ripples; the draft's description now says
  the EEG is not included.
- The age is "/" and the pixel scale and camera view are not described (partly fixed in the draft).
  Still present, without the draft's statements that they were not recorded.
- Rat 2's preflight units share no tetrode with its flight units. Still present (0 tetrodes in
  common, against 4 to 6 for Rats 1 and 3).

The first attempt's other findings, which `001754-knierim-2000-02` found fixed in the draft, are
present in this version: `cell_number` does not link maps to units (84 of 211, and -1 for every
Rat3 FD4 ES map); -999 for unvisited bins and the lost position counted in bin [63, 0] are not
described; six groups of identical maps; spike-time resolution and zero intervals not described;
and Rat1 FD4's overlapping baseline epochs.

## Spatial information in flight against preflight

The three comparisons (`si-comparable-to-preflight`, `si-rat1-fd9-vs-preflight`,
`si-rat3-fd4-vs-preflight`) share their method, from the claims file: active cells (mean rate above
0.05 Hz on the track), Skaggs' information per spike with off-track positions kept, flight against
the same rat's preflight cells by a two-sided Mann-Whitney test at p < 0.05, run under every reading.

How the readings apply here:

- M-a, the deposited maps. Each map, identical copies counted once, over its visited bins, without
  the lost-position bin. A map is active when its occupancy-weighted mean rate exceeds 0.05 Hz, as
  maps cannot be linked to units in this version (data check).
- M-b, maps recomputed from spikes and position. Position exists only at spike times and only in
  flight, so spike counts are divided by the deposited occupancy map of the epoch (asked in
  `questions/001754.yaml`), and M-b cannot be computed for any preflight cell. A unit is active when
  its spikes over the epoch's duration exceed 0.05 Hz. M-b gives flight values only; they are shown,
  with a test of M-b flight values against M-a preflight values as a variant that the criterion does
  not name.
- S-a and S-b. Each preflight file is one recording day with one ES and one MC epoch, so S-a has two
  candidates, ES1 and MC1. S-b, the session closest before launch, is that day; within it MC1 is the
  later epoch, so S-b is read as MC1 and adds no reading beyond S-a (asked). Each flight session has
  one ES epoch (for rat 3, session 2 is ES1).

The criterion's rule is applied to the M-a readings, the only ones with preflight values.

In [ ]:
# Reading M-a: every deposited map, identical copies counted once, active when its occupancy-weighted
# mean rate exceeds 0.05 Hz. Reading M-b (flight only): every unit, active when its spikes in the epoch
# over the epoch's duration exceed 0.05 Hz. Active units by spike count are also given for preflight.
ma = nl.ma_table(sessions)
print("Deposited maps per file and epoch: all, identical copies, at or below 0.05 Hz (of those counted once), active:")
print(ma.groupby(["rat", "day", "epoch"]).agg(maps=("info", "size"), copies=("duplicate", "sum"),
      at_or_below_threshold=("mean_rate", lambda r: int(((r <= nl.ACTIVE_HZ) & ~ma.loc[r.index, "duplicate"]).sum())),
      active=("active", "sum")).to_string())
active_ma = ma[ma.active]
print("\nSpatial information of active M-a maps, range:", round(active_ma["info"].min(), 3), "to", round(active_ma["info"].max(), 3), "bits/spike")

rows = []
for s in sessions:
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ == "BL":
            continue
        rates = np.array([((sp >= a) & (sp <= b)).sum() / (b - a) for sp in s["units"].spikes])
        rows.append({"rat": s["subject"], "day": s["day"], "epoch": name, "units": len(rates), "active_units_by_spikes": int((rates > nl.ACTIVE_HZ).sum()),
                     "active_maps_M-a": int(active_ma[(active_ma.rat == s["subject"]) & (active_ma.day == s["day"]) & (active_ma.epoch == name)].shape[0])})
active_counts = pd.DataFrame(rows)
print("\nActive cells per epoch: units by spike count, and deposited maps (M-a):")
print(active_counts.to_string(index=False))

mb = {}
for s in sessions:
    if "xy" not in s:
        continue
    for name in s["epochs"].name[s["epochs"].type != "BL"]:
        mb[(s["subject"], s["day"], name)] = nl.mb_maps(s, name)
print("\nReading M-b, active units per flight epoch: spikes, spikes dropped at the lost position or with no sample, spikes in unvisited bins:")
print(pd.DataFrame([{"rat": k[0], "day": k[1], "epoch": k[2], "active": int(d.active.sum()), "spikes": int(d[d.active].spikes.sum()),
                     "dropped_lost_or_no_sample": int(d[d.active].spikes_lost_position.sum()), "in_unvisited_bins": int(d[d.active].spikes_unvisited_bin.sum()),
                     "info_range": f"{d[d.active]['info'].min():.3f}-{d[d.active]['info'].max():.3f}"} for k, d in mb.items()]).to_string(index=False))
print("The preflight files hold no position, so M-b cannot be computed for any preflight cell.")

def ma_values(rat, day, epoch):
    return active_ma[(active_ma.rat == rat) & (active_ma.day == day) & (active_ma.epoch == epoch)]["info"].values

def mb_values(rat, day, epoch):
    d = mb[(rat, day, epoch)]
    return d[d.active]["info"].dropna().values

def readings(rat, day, epoch):
    """Every reading of a flight-against-preflight comparison: M-a with each preflight epoch (S-a;
    S-b is the later one, MC1), and as variants M-b flight values against M-a preflight values."""
    rows = []
    for pre in ["ES1", "MC1"]:
        rows.append({"reading": f"M-a, preflight {pre}" + (" (S-a; S-b)" if pre == "MC1" else " (S-a)")} | nl.compare(ma_values(rat, day, epoch), ma_values(rat, "preflight", pre)))
    for pre in ["ES1", "MC1"]:
        rows.append({"reading": f"variant: M-b flight, M-a preflight {pre}"} | nl.compare(mb_values(rat, day, epoch), ma_values(rat, "preflight", pre)))
    return pd.DataFrame(rows)

def within(df, expected):
    """Whether each expected value is within its tolerance, per reading."""
    (mf, tf), (mp, tp), (nf, tnf), (np_, tnp) = expected
    return df.assign(flight_mean_ok=(df.mean_flight - mf).abs() <= tf, pre_mean_ok=(df.mean_pre - mp).abs() <= tp,
                     n_flight_ok=(df.n_flight - nf).abs() <= tnf, n_pre_ok=(df.n_pre - np_).abs() <= tnp)

def judge_absence(df):
    """The criterion's rule on the readings of the criterion (M-a rows; M-b has no preflight values)."""
    d = df[df.reading.str.startswith("M-a")]
    if not d.significant.any():
        return "reproduced" if (d.flight_mean_ok & d.pre_mean_ok).any() else "partially_reproduced"
    if d.significant.all():
        return "not_reproduced"
    return "blocked (methods_underspecified)"

show = ["reading", "n_flight", "n_pre", "mean_flight", "sem_flight", "mean_pre", "sem_pre", "difference", "U", "p", "significant",
        "flight_mean_ok", "pre_mean_ok", "n_flight_ok", "n_pre_ok"]

### si-comparable-to-preflight

> For rat 2, the spatial information per spike of the 16 active cells recorded on the Escher
> staircase on flight day 9 (1.13 +/- 0.09 bits) did not differ significantly from that of its 19
> active preflight cells (1.12 +/- 0.08 bits), by Mann-Whitney test.

Reproduced when no reading finds a difference and at least one puts both means within tolerance;
partially reproduced when none finds a difference and no reading puts both means within tolerance.

In [ ]:
si2 = within(readings("Rat2", "FD9", "ES1"), [(1.13, 0.09), (1.12, 0.08), (16, 1.6), (19, 1.9)])
print("Rat2, FD9 ES1 against preflight (expected 1.13 +/- 0.09 and 1.12 +/- 0.08 bits/spike, n 16 and 19):")
print(si2[show].round(4).to_string(index=False))
outcome_si2 = judge_absence(si2)
print("\nBy the criterion's rule:", outcome_si2)

**Result: partially reproduced.** Under both readings the test finds no difference: Rat 2's 17
active FD9 ES maps (1.377 +/- 0.120 bits/spike) against its 21 preflight ES maps (1.320 +/- 0.112,
p = 0.88, difference 0.057) and its 25 preflight MC maps (1.284 +/- 0.116, p = 0.66, difference
0.092). No reading puts both means within tolerance: the FD9 mean (1.377) is above 1.13 + 0.09, and
the preflight means (1.320 and 1.284) are above 1.12 + 0.08. The FD9 count (17) is within 16 +/-
1.6; the preflight counts (21 and 25) are not within 19 +/- 1.9. Under M-b the 25 active FD9 units
have 1.6 to 1.7 bits/spike more than either preflight M-a set (2.965 +/- 0.224, p < 1e-6), but M-b
has no preflight values, so this mixed test is not a reading of the criterion. The FD9 ES window is
the one in which the trace behaves as Rat 2's position (data check). Rat 2's preflight units share
no tetrode with its flight units, and which preflight epoch was the rectangular track of Fig. 2b is
not stated.

### si-rat1-fd9-vs-preflight

> For rat 1, the mean spatial information of its 21 active cells on flight day 9 (1.19 +/- 0.09
> bits) did not differ significantly from its preflight cells (1.37 +/- 0.10 bits, n = 28).

As for `si-comparable-to-preflight`, for rat 1; blocked as methods_underspecified when the readings
disagree on the test.

In [ ]:
si1 = within(readings("Rat1", "FD9", "ES1"), [(1.19, 0.09), (1.37, 0.10), (21, 2.1), (28, 2.8)])
print("Rat1, FD9 ES1 against preflight (expected 1.19 +/- 0.09 and 1.37 +/- 0.10 bits/spike, n 21 and 28):")
print(si1[show].round(4).to_string(index=False))
outcome_si1 = judge_absence(si1)
print("\nBy the criterion's rule:", outcome_si1)

**Result: blocked (methods underspecified).** The readings disagree on the test. Against the
preflight ES epoch, Rat 1's 24 active FD9 ES maps (1.588 +/- 0.143 bits/spike) do not differ from its
25 preflight maps (1.804 +/- 0.149, p = 0.50, difference -0.216); against the preflight MC epoch,
which is also S-b, they are higher than its 24 maps (1.127 +/- 0.114, p = 0.0055, difference 0.461).
The paper does not say on which track, or in which epoch, rat 1's preflight cells were recorded.
No reading puts a mean within tolerance (1.588 against 1.19 +/- 0.09; 1.804 and 1.127 against
1.37 +/- 0.10), and no count is within 10% of the paper's (24 against 21; 25 and 24 against 28).
The M-b variant has 3.331 +/- 0.268 bits/spike over 24 active units.

### si-rat3-fd4-vs-preflight

> For rat 3, the mean spatial information of all 7 active place cells on flight day 4 (1.12 +/- 0.27
> bits, session 2) was not different from its preflight cells (1.48 +/- 0.11 bits, n = 24).

As for `si-comparable-to-preflight`, for rat 3, with the FD4 cells from session 2 (ES1).

In [ ]:
si3 = within(readings("Rat3", "FD4", "ES1"), [(1.12, 0.27), (1.48, 0.11), (7, 0.7), (24, 2.4)])
print("Rat3, FD4 ES1 (session 2) against preflight (expected 1.12 +/- 0.27 and 1.48 +/- 0.11 bits/spike, n 7 and 24):")
print(si3[show].round(4).to_string(index=False))
outcome_si3 = judge_absence(si3)
print("\nBy the criterion's rule:", outcome_si3)
si3b = within(readings("Rat3", "FD4", "ES2"), [(1.12, 0.27), (1.48, 0.11), (7, 0.7), (24, 2.4)])
print("\nVariant, not the claim's session: FD4 ES2 (session 3) against preflight:")
print(si3b[show].round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
groups = [(f"{r}\n{d} {e}", ma_values(r, d, e)) for r, d, e in [("Rat1", "preflight", "ES1"), ("Rat1", "preflight", "MC1"), ("Rat1", "FD9", "ES1"),
          ("Rat2", "preflight", "ES1"), ("Rat2", "preflight", "MC1"), ("Rat2", "FD9", "ES1"), ("Rat3", "preflight", "ES1"), ("Rat3", "preflight", "MC1"), ("Rat3", "FD4", "ES1")]]
ax.boxplot([v for _, v in groups], showfliers=True)
ax.set_xticks(range(1, len(groups) + 1), [g for g, _ in groups], fontsize=7)
ax.set_ylabel("spatial information, active M-a maps (bits/spike)")
plt.tight_layout()
plt.show()
print("All plotted values are within the axes; range", round(min(v.min() for _, v in groups), 3), "to", round(max(v.max() for _, v in groups), 3))

**Result: reproduced.** Under both readings the test finds no difference: Rat 3's 11 active
session 2 maps (1.231 +/- 0.206 bits/spike) against its 38 preflight ES maps (1.518 +/- 0.125, p =
0.28, difference -0.287) and its 33 preflight MC maps (1.337 +/- 0.121, p = 0.77, difference
-0.106). Under the ES reading both means are within tolerance (1.231 within 1.12 +/- 0.27, 1.518
within 1.48 +/- 0.11), as the criterion requires of one reading. The counts are not: 11 active maps
in session 2 against the paper's 7, and 38 or 33 preflight against 24, so the cells compared are not
the paper's set; the criterion's rule does not use the counts. Session 3 (ES2, 19 maps, 1.384 +/-
0.177) also does not differ from either preflight epoch (p = 0.28 and 0.86). The M-b variant has
3.038 +/- 0.334 bits/spike over 16 active units.

## flight-spatial-code

> In flight, after an initial period, hippocampal place cells on the Escher staircase showed
> normal, spatially specific firing, with spatial information per spike not different from the same
> rats' preflight cells.

Judged from the three comparisons above: reproduced when all three are reproduced; partially
reproduced when both FD9 comparisons are reproduced and the FD4 one is not, or when no FD9
comparison is not reproduced and one or more is partially reproduced; not reproduced when either
FD9 comparison is not reproduced. When a dependency is blocked as methods_underspecified, so is
this claim, unless the others already decide it.

In [ ]:
OUTCOMES = ["reproduced", "partially_reproduced", "not_reproduced"]

def flight_rule(r2, r1, r3):
    if r2 == r1 == r3 == "reproduced":
        return "reproduced"
    if "not_reproduced" in (r2, r1):
        return "not_reproduced"
    if r2 == r1 == "reproduced" and r3 != "reproduced":
        return "partially_reproduced"
    return "partially_reproduced"

deps = {"si-comparable-to-preflight": outcome_si2, "si-rat1-fd9-vs-preflight": outcome_si1, "si-rat3-fd4-vs-preflight": outcome_si3}
print("Dependencies:", deps)
options = [[o] if o in OUTCOMES else OUTCOMES for o in deps.values()]
possible = sorted({flight_rule(a, b, c) for a in options[0] for b in options[1] for c in options[2]})
print("Outcomes the rule gives over every value a blocked dependency could take:", possible)
outcome_flight = possible[0] if len(possible) == 1 else "blocked (methods_underspecified)"
print("By the criterion's rule:", outcome_flight)

**Result: blocked (methods underspecified).** The rat 2 comparison is partially reproduced, the rat
3 comparison reproduced, and the rat 1 comparison blocked. The others do not decide the claim: if the
rat 1 comparison were reproduced or partially reproduced the claim would be partially reproduced,
and if it were not reproduced the claim would be not reproduced. Under the preflight ES reading for
rat 1 (p = 0.50) the claim would be partially reproduced, and under the preflight MC reading (p =
0.0055, flight higher) not reproduced. In `001754-knierim-2000-02` this claim was not reproduced
by a shuffle test the paper does not use; that test is not run here.

## fd4-abnormal-selectivity

> On the first experience of the Escher staircase on flight day 4, place cells of rats 1 and 2
> showed abnormal patterns of spatial selectivity, which differed between the two rats.

Abnormal is read as spatial information of active cells on FD4 significantly lower than preflight
(two-sided p < 0.05), in each of rats 1 and 2, with the map and session readings. Reproduced when this
holds for both rats under every reading; partially reproduced when it holds for one rat; not
reproduced when it holds for neither; blocked when readings disagree. M-b has no preflight values,
so the readings are M-a against preflight ES1 and MC1.

In [ ]:
rows = []
for rat in ["Rat1", "Rat2", "Rat3"]:
    for pre in ["ES1", "MC1"]:
        r = nl.compare(ma_values(rat, "FD4", "ES1"), ma_values(rat, "preflight", pre))
        rows.append({"rat": rat, "reading": f"M-a, preflight {pre}" + (" (S-a; S-b)" if pre == "MC1" else " (S-a)")} | r |
                    {"lower_and_significant": r["significant"] and r["difference"] < 0})
fd4 = pd.DataFrame(rows)
print("FD4 ES1 against preflight, two-sided Mann-Whitney (Rat3 is not part of the claim and is shown for reference):")
print(fd4[["rat", "reading", "n_flight", "n_pre", "mean_flight", "sem_flight", "mean_pre", "sem_pre", "difference", "U", "p", "lower_and_significant"]].round(4).to_string(index=False))
per_rat = {rat: set(fd4[fd4.rat == rat].lower_and_significant) for rat in ["Rat1", "Rat2"]}
print("\nAbnormal (lower and significant) under the readings, per rat:", {k: sorted(v) for k, v in per_rat.items()})
if any(len(v) > 1 for v in per_rat.values()):
    outcome_fd4 = "blocked (methods_underspecified)"
else:
    holds = sum(v == {True} for v in per_rat.values())
    outcome_fd4 = {2: "reproduced", 1: "partially_reproduced", 0: "not_reproduced"}[holds]
print("By the criterion's rule:", outcome_fd4)
print("\nVariant, M-b FD4 ES1 values (no preflight M-b exists): mean +/- s.e.m. and n of active units")
for rat in ["Rat1", "Rat2"]:
    v = mb_values(rat, "FD4", "ES1")
    print(f"  {rat}: {np.mean(v):.3f} +/- {stats.sem(v):.3f}, n {len(v)}; against FD9 ES1 M-b {np.mean(mb_values(rat, 'FD9', 'ES1')):.3f}, "
          f"Mann-Whitney p {stats.mannwhitneyu(v, mb_values(rat, 'FD9', 'ES1')).pvalue:.3g}")
print("Variant, M-a FD4 ES1 against FD9 ES1 (not a reading of the criterion):")
for rat in ["Rat1", "Rat2"]:
    print(f"  {rat}:", nl.compare(ma_values(rat, "FD4", "ES1"), ma_values(rat, "FD9", "ES1")))

**Result: partially reproduced.** It holds for rat 2 and not for rat 1, under both readings. Rat 2's
14 active FD4 ES maps (0.708 +/- 0.112 bits/spike) are lower than its preflight ES maps (1.320 +/-
0.112, p = 0.00063) and MC maps (1.284 +/- 0.116, p = 0.0015). Rat 1's 20 FD4 ES maps (1.409 +/-
0.192) are lower than its preflight ES maps without significance (1.804 +/- 0.149, p = 0.077) and
higher than its MC maps (1.127 +/- 0.114, p = 0.36). Rat 2's FD4 maps are also lower than its FD9
maps (1.377, p = 0.00033), and Rat 1's are not (1.588, p = 0.38). In Rat 2's FD4 ES window the
trace could not be shown to be Rat 2's position (data check), and that file holds five identical
copies of maps (each counted once). That the patterns differed between the rats has no measure and
is not judged; Rat 2's FD4 maps carry less information than Rat 1's (0.708 against 1.409). The M-b
FD4 values (4.037 and 3.739 bits/spike) have no preflight counterpart.

## fig3-stability-across-sessions

> For rat 3 on flight day 4, most of the 7 cells kept the same firing fields in a third session
> that followed the second immediately; the fourth cell lost its field, and a few other cells
> gained a field in session 3.

Cells are the active cells of session 2 (ES1), followed into session 3 (ES2). A cell keeps its
field by (a) when the correlation of its session 2 and 3 maps over bins visited in both exceeds
the 95th percentile of the correlations between different cells across the two sessions, and by
(b) when its session 3 peak lies in its session 2 field (bins with at least 20% of the session 2
peak). "Most" is read as at least 4 of the 7; reproduced when at least 4 keep their field under
every reading, not reproduced when 3 or fewer do under every reading, and blocked when the readings
disagree.

Reading M-b uses the unsmoothed maps of the active units. Reading M-a needs the deposited maps of
one cell in both sessions, and the dandiset gives no link (every Rat3 FD4 ES map has
`cell_number` -1), so M-a is run as a variant, with each deposited map paired to the unit whose
spikes it matches best. The number of active cells is not 7, so "at least 4 of the 7" is run both as
a count of 4 and as a fraction of 4/7 (asked).

In [ ]:
s3 = by_label["Rat3 FD4"]
occ2, occ3 = (nl.clean_occupancy(nl.epoch_occupancy(s3, e)) for e in ["ES1", "ES2"])
both = (occ2 > 0) & (occ3 > 0)
print("Rat3 FD4: ES1 (session 2)", s3["epochs"].set_index("name").loc["ES1", ["start", "stop"]].tolist(), "s; ES2 (session 3)",
      s3["epochs"].set_index("name").loc["ES2", ["start", "stop"]].tolist(), "s; bins visited in both:", int(both.sum()))

def stability(maps2, maps3, cells, fraction=nl.FIELD_FRACTION):
    """Readings (a) and (b) for the cells; maps2/maps3 map a cell to its session 2 and 3 maps (None when absent)."""
    C_ = np.array([[nl.map_correlation(maps2[i], maps3[j], both) if maps3[j] is not None else np.nan for j in cells] for i in cells])
    off = C_[~np.eye(len(cells), dtype=bool)]
    thr = float(np.percentile(off[np.isfinite(off)], 95))
    same = np.diag(C_)
    peak = [maps3[i] is not None and nl.peak_in_field(maps2[i], maps3[i], both, fraction) for i in cells]
    return pd.DataFrame({"cell": cells, "r_same_cell": same, "keeps_a": np.nan_to_num(same, nan=-2) > thr, "keeps_b": peak}), thr, len(off[np.isfinite(off)])

def judge_most(n_keep, n_cells):
    return {"count >= 4": n_keep >= 4, "fraction >= 4/7": n_keep / n_cells >= 4 / 7}

# Reading M-b: the active units of session 2 (spike count over ES1's duration above 0.05 Hz).
b2, b3 = mb[("Rat3", "FD4", "ES1")], mb[("Rat3", "FD4", "ES2")]
cells_b = list(b2.index[b2.active])
maps2_b = {i: b2.rate_map[i] for i in cells_b}
maps3_b = {i: (b3.rate_map[i] if b3.spikes[i] > 0 else None) for i in cells_b}
tab_b, thr_b, n_null_b = stability(maps2_b, maps3_b, cells_b)
tab_b = tab_b.assign(tetrode=[b2.tetrode[i] for i in cells_b], cluster=[b2.cluster[i] for i in cells_b],
                     spikes_s2=[b2.spikes[i] for i in cells_b], spikes_s3=[b3.spikes[i] for i in cells_b], rate_s3_hz=[round(b3.mean_rate[i], 3) for i in cells_b])
print(f"\nM-b: {len(cells_b)} active cells in session 2 (expected 7 +/- 0.7); null 95th percentile of different-cell correlations {thr_b:.3f} over {n_null_b} pairs")
print(tab_b[["tetrode", "cluster", "spikes_s2", "spikes_s3", "rate_s3_hz", "r_same_cell", "keeps_a", "keeps_b"]].round(3).to_string(index=False))
print(f"Keep their field: (a) {int(tab_b.keeps_a.sum())} of {len(tab_b)}, (b) {int(tab_b.keeps_b.sum())} of {len(tab_b)}")
for name, k in [("(a)", int(tab_b.keeps_a.sum())), ("(b)", int(tab_b.keeps_b.sum()))]:
    print(f"  {name} 'most' as:", judge_most(k, len(tab_b)))
print("Same-cell correlations, range:", round(np.nanmin(tab_b.r_same_cell), 3), "to", round(np.nanmax(tab_b.r_same_cell), 3))
print("\nField threshold variants for reading (b):", {f: int(stability(maps2_b, maps3_b, cells_b, f)[0].keeps_b.sum()) for f in [0.1, 0.2, 0.3, 0.5]})

In [ ]:
# Variant for reading M-a. The dandiset gives no link between a deposited map and a unit here (every
# Rat3 FD4 ES map has cell_number -1), so each unit is paired with the map of its tetrode and epoch whose
# rate x occupancy best matches its spike-count map (data check). This pairing is the attempt's own.
def best_map(s, epoch, ui):
    a, b = s["epochs"].set_index("name").loc[epoch, ["start", "stop"]]
    u = s["units"].loc[ui]
    ok, r, c = nl.spike_positions(s, u.spikes[(u.spikes >= a) & (u.spikes <= b)])
    cand = s["maps"][(s["maps"].epoch == epoch) & ~s["maps"].duplicate & (s["maps"].tetrode == u.tetrode)]
    if ok.sum() == 0 or not len(cand):
        return None, np.nan
    counts = nl.count_map(r, c)
    rs = {i: np.corrcoef((np.clip(s["rate_maps"][i], 0, None) * nl.clean_occupancy(s["occupancy_maps"][i])).ravel(), counts.ravel())[0, 1] for i in cand.index}
    rs = {i: v for i, v in rs.items() if np.isfinite(v)}
    return (max(rs, key=rs.get), max(rs.values())) if rs else (None, np.nan)

def deposited(i):
    m = s3["rate_maps"][i].copy()
    m[m == nl.UNVISITED] = np.nan
    return m

a3 = ma[(ma.rat == "Rat3") & (ma.day == "FD4")]
cells_a = list(a3[(a3.epoch == "ES1") & a3.active].row)
owner = {}
for ui in s3["units"].index:
    i, r = best_map(s3, "ES1", ui)
    if i is not None and (i not in owner or r > owner[i][1]):
        owner[i] = (ui, r)
maps2_a, maps3_a, pairs = {}, {}, []
for i in cells_a:
    ui = owner.get(i, (None,))[0]
    j = best_map(s3, "ES2", ui)[0] if ui is not None else None
    maps2_a[i], maps3_a[i] = deposited(i), (deposited(j) if j is not None else None)
    pairs.append({"map_s2": s3["maps"].source_file[i], "tetrode": s3["maps"].tetrode[i], "unit_cluster": s3["units"].cluster[ui] if ui is not None else None,
                  "r_link_s2": round(owner[i][1], 3) if ui is not None else np.nan, "map_s3": s3["maps"].source_file[j] if j is not None else None})
tab_a, thr_a, n_null_a = stability(maps2_a, maps3_a, cells_a)
print(f"M-a variant: {len(cells_a)} active deposited maps in session 2; null 95th percentile {thr_a:.3f} over {n_null_a} pairs")
print(pd.concat([pd.DataFrame(pairs), tab_a[["r_same_cell", "keeps_a", "keeps_b"]]], axis=1).round(3).to_string(index=False))
print(f"Keep their field: (a) {int(tab_a.keeps_a.sum())} of {len(tab_a)}, (b) {int(tab_a.keeps_b.sum())} of {len(tab_a)}")
for name, k in [("(a)", int(tab_a.keeps_a.sum())), ("(b)", int(tab_a.keeps_b.sum()))]:
    print(f"  {name} 'most' as:", judge_most(k, len(tab_a)))

# Cells silent or below 0.05 Hz in session 2 that are active in session 3 (reported, not judged).
gained = b3[b3.active & ~b2.active]
print(f"\nUnits below 0.05 Hz in session 2 and active in session 3: {len(gained)}; their spikes in session 2: {b2.spikes[gained.index].tolist()}, in session 3: {gained.spikes.tolist()}")
print("Units active in session 2 and silent (no spike) in session 3:", int((b2.active & (b3.spikes == 0)).sum()))
results_fig3 = {(m, r): v for m, tab in [("M-b", tab_b), ("M-a variant", tab_a)] for r in ["a", "b"]
                for v in [judge_most(int(tab[f"keeps_{r}"].sum()), len(tab))]}
by_reading = {f"{m} ({r}), {k}": v for (m, r), d in results_fig3.items() for k, v in d.items()}
print("\nDoes 'most' hold, per reading:", by_reading)

**Result: blocked (methods underspecified).** Session 2 has 16 active cells under M-b, not 7 (11
deposited maps under M-a), so the criterion's "at least 4 of the 7" has two readings, and they give
different outcomes. Under M-b, 6 of the 16 cells keep their field by reading (a) (same-cell
correlations -0.024 to 0.366 against a 95th percentile of 0.094 over 240 different-cell pairs) and
1 of 16 by reading (b). Read as a count of at least 4, (a) holds and (b) does not, so the readings
disagree; read as a fraction of at least 4/7, neither holds (0.38 and 0.06), which would give not
reproduced. Reading (b) gives 2, 1, 0 and 0 cells at field thresholds of 0.1, 0.2, 0.3 and 0.5. In
the M-a variant, with the attempt's own pairing of maps to units, 6 and 5 of 11 cells keep their
field by (a) and (b) (95th percentile 0.253): at least 4 as a count, below 4/7 as a fraction.
Four units below 0.05 Hz in session 2 are active in session 3 (two of them silent in session 2),
which fits the legend's cells that gained a field; no active cell of session 2 is silent in session
3. In `001754-knierim-2000-02` this claim was reproduced by a median correlation of smoothed maps,
a criterion the paper does not give.

## direction-selectivity

> Some cells had place fields only when the rat moved counterclockwise through a location, showing
> the direction selectivity seen on such tracks on Earth.

Blocked: data missing. The criterion forms, for each bin, a counterclockwise rate and the clockwise
spike count expected from it, which need the time the rat spent in the bin moving in each direction.
The dandiset gives position only at spike times, irregularly (7.6 to 35.7 samples per second in the
ES epochs, with gaps up to 12.7 s, printed below), and one occupancy map per epoch with no direction.
Nothing in the files or the conversion code gives the track's layout in the image, from which the
direction along the circuit could be read. The claims file names no outside data for this claim,
and the NASA record (below) lists no continuous position. Asked in `questions/001754.yaml`.

In [ ]:
# What the criterion's rates per direction would need: time spent in each bin in each direction. The
# position is sampled only at spike times; the gaps between samples in the flight ES windows, and the
# occupancy maps, one per epoch, with no direction.
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, name in s["epochs"][s["epochs"].type == "ES"][["start", "stop", "name"]].itertuples(index=False):
        dt = np.diff(s["t"][(s["t"] >= a) & (s["t"] <= b)])
        rows.append({"session": nl.label(s), "epoch": name, "samples_per_s": round((len(dt) + 1) / (b - a), 1), "median_gap_s": round(float(np.median(dt)), 4),
                     "gaps_over_1s": int((dt > 1).sum()), "longest_gap_s": round(float(dt.max()), 1),
                     "occupancy_maps_for_epoch": len({s["occupancy_maps"][i].tobytes() for i in s["maps"].index[s["maps"].epoch == name]})})
print(pd.DataFrame(rows).to_string(index=False))

## theta-normal and ripples-normal

> Hippocampal EEG showed normal theta rhythm during active locomotion, in all rats on flight days 4
> and 9.

> Hippocampal EEG showed normal sharp waves and ripples during quiet inactivity in the sleeping
> pouch, on both flight days.

Blocked: data missing. No file holds EEG or any continuous voltage (data check: no ElectricalSeries
or LFP in any file). The NASA record of the study (OSD-968) has no passage on EEG, theta, ripples
or LFP, and none of the names inside its zip files suggests EEG (below), so the EEG is not public
there either, as far as its names show; the zip files were not opened beyond their directories.

In [ ]:
# The NASA record of the study (OSD-968): what its metadata says of EEG and of the cameras, its files, and
# the names inside each zip file (read from each zip's directory, by range requests; nothing is saved).
osd_text = json.dumps(requests.get("https://osdr.nasa.gov/osdr/data/osd/meta/968", timeout=60).json(), ensure_ascii=False)
for word in ["EEG", "theta", "ripple", "LFP", "camera"]:
    hits = [m.group(0) for m in re.finditer(r"[^.\"]{0,150}" + re.escape(word) + r"[^\"]{0,300}", osd_text)]
    print(f"OSD-968 metadata, '{word}': {len(hits)} passage(s)")
    for h in hits[:1]:
        print("   ", h[:450])
osd_files = requests.get("https://osdr.nasa.gov/osdr/data/osd/files/968", timeout=60).json()["studies"]["OSD-968"]["study_files"]
print(f"\nOSD-968 files ({len(osd_files)}):")
eeg_like = re.compile(r"eeg|lfp|csc|theta|ripple|\.ncs$|\.eeg$", re.I)
for f in osd_files:
    line = f"  {f['file_name']}  {f['file_size'] / 1e6:.1f} MB"
    if f["file_name"].endswith(".zip"):
        url = requests.head("https://osdr.nasa.gov" + f["remote_url"], allow_redirects=True, timeout=60).url
        names = [n for n in zipfile.ZipFile(remfile.File(url)).namelist() if "__MACOSX" not in n]
        line += f"; {len(names)} entries, {sum(bool(eeg_like.search(n)) for n in names)} with an EEG-like name"
    print(line)

## three-orthogonal-surfaces

> Hippocampal cells formed distinct, reliable representations of position on the three orthogonal
> surfaces of the Escher staircase in microgravity.

Blocked (methods underspecified). Part (i) needs `flight-spatial-code` and
`fig3-stability-across-sessions` reproduced, and both are blocked as methods_underspecified, so this
claim is blocked for the same reason. Part (ii) could not be tested in any case: it needs the
track's three surfaces drawn in the position frame, and the dandiset holds one two-dimensional pixel
position with no camera view, track layout or pixel scale (data check), while the NASA record says
position was recorded with two synchronised cameras (printed above). The conversion code that made
these files, read by the first attempt at commit b538d9e, describes the ES track only as a
three-dimensional track with 90 degree turns and the position as pixel coordinates (lines below).

In [ ]:
repo, commit = "catalystneuro/mcnaughton2000-to-nwb", "b538d9e07079394362a64e5118a226e91bdd7ab5"
tree = requests.get(f"https://api.github.com/repos/{repo}/git/trees/{commit}?recursive=1", timeout=60).json().get("tree", [])
paths = [t["path"] for t in tree if t["type"] == "blob" and t["path"].endswith((".py", ".md"))]
print(f"Conversion code at {commit[:7]} ({len(paths)} .py and .md files), every line on the track, camera, surfaces or coordinates:")
for path in paths:
    text = requests.get(f"https://raw.githubusercontent.com/{repo}/{commit}/{path}", timeout=60).text
    for n, line in enumerate(text.splitlines(), 1):
        if re.search(r"camera|surface|staircase|layout|geometry|reference_frame|coordinate", line, re.I):
            print(f"  {path}:{n}: {line.strip()[:160]}")

## Summary

In [ ]:
print(f"Run time: {(time.time() - started) / 60:.1f} minutes")

| Claim | Outcome | In `001754-knierim-2000-02` |
|---|---|---|
| `flight-spatial-code` | Blocked: methods underspecified | Not reproduced (shuffle test) |
| `si-comparable-to-preflight` | Partially reproduced | Reproduced (removed from that record later) |
| `si-rat1-fd9-vs-preflight` | Blocked: methods underspecified | New claim |
| `si-rat3-fd4-vs-preflight` | Reproduced | New claim |
| `fd4-abnormal-selectivity` | Partially reproduced | Reproduced (removed from that record later) |
| `fig3-stability-across-sessions` | Blocked: methods underspecified | Reproduced |
| `direction-selectivity` | Blocked: data missing | New claim |
| `theta-normal` | Blocked: data missing | Blocked: data missing (removed later) |
| `ripples-normal` | Blocked: data missing | Blocked: data missing (removed later) |
| `three-orthogonal-surfaces` | Blocked: methods underspecified | Blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper gives no bin size, smoothing or occupancy rule;
the deposited maps (64 x 64, smoothed with an unstated kernel) are used as the authors' maps (M-a),
and recomputed maps (M-b) can only be made for flight, with the deposited occupancy. M-a counts a map
as active by its own mean rate, as maps cannot be linked to units. The active-cell counts differ
from the paper's in most comparisons (for example 11 against 7 for rat 3 in session 2, 38 against
24 for its preflight ES maps), so the cells are not the paper's set, and the spatial information of
the deposited maps is above the paper's means for rat 2 and for rat 1's FD9 and preflight ES maps.
Which preflight epoch the paper used is not stated; for rat 1 the choice decides the test. The
stability claim's count rule was written for 7 cells.

What was found about the dandiset (version 0.260728.1352). The Rat1 and Rat2 files of each flight
day hold one tracker's trace, presented as each rat's position; it behaves as Rat 2's position in
Rat 2's FD9 ES window and could not be attributed in Rat 2's FD4 ES window. Position is given only
at spike times, none preflight. Rate maps cannot be linked to units: `cell_number` names the best
match for 84 of 211 unit-epochs and is -1 for every Rat3 FD4 ES map. The maps are smoothed with an
unstated kernel, use -999 for unvisited bins and count the lost position in bin [63, 0], without
saying so, and six groups of maps are stored twice or more. The dandiset description reports EEG,
which no file holds, and places the preflight recordings on a rectangular track while the files
label them ES and MC; the files' experiment description puts the rectangular track in the baseline
sessions, which the paper places in the sleeping pouch. Rat 2's preflight and flight units share no
tetrode. Ages, pixel scale and camera view are missing. The draft fixes several of these; it is not
published.